# Introduction to Machine Learning: Supervised Learning

**Instructor:** Daniel Acuna, Ph.D.
**Position:** Associate Professor of Computer Science
**Institution:** University of Colorado Boulder

---

Lab 1: Introduction to Machine Learning: Supervised Learning

---

## Setup (do not edit)

In [3]:
import pathlib
from typing import Tuple, Dict, List

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression

RANDOM_STATE: int = 42  # global seed for full reproducibility
np.random.seed(RANDOM_STATE)

_DATA_PATH = pathlib.Path("california_housing.csv")
if not _DATA_PATH.exists():
    raise FileNotFoundError(
        "california_housing.csv is missing from the lab directory. Please download it or ask the TA "
        "for assistance."
    )

## 1. Load the dataset *(10 points)*

Write a function `load_housing()` that reads the CSV into a `pandas.DataFrame`.  Store the shape of
that DataFrame in a variable called **`q1_shape`**.

In [4]:


def load_housing() -> pd.DataFrame:
    """Load the California Housing data from ``california_housing.csv``.

    Returns
    -------
    pd.DataFrame
        A DataFrame containing the 8 predictors plus the target column ``MedHouseVal``.
    """
    # your code here
#     raise NotImplementedError
    return pd.read_csv('california_housing.csv')


# Compute the answer required by the autograder
q1_shape: Tuple[int, int] = load_housing().shape

In [5]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Exercise 5.1
assert isinstance(q1_shape, tuple), "q1_shape must be a tuple"
assert len(q1_shape) == 2, "q1_shape should have 2 elements (rows, cols)"
assert q1_shape[0] > 0 and q1_shape[1] > 0, "Shape values must be positive"
print(f"Dataset shape: {q1_shape}")


Dataset shape: (20640, 9)


## 2. Income → Value gap *(10 points)*

Split the dataset into **income quartiles** using the ``MedInc`` feature:
* bottom 25 % (Q1)
* top 25 %  (Q4)

Compute the **mean** of the target variable ``MedHouseVal`` for each of those two groups and store
their *difference* (**top – bottom**) in **`q2_income_value_gap`** (float, rounded to three
decimals).

In [6]:

df = load_housing()

print(df.describe())

print(df.head())

# get Series to get quantile value for splitting data
med_inc = df['MedInc']
med_inc
print(med_inc.describe())

# ref @ https://pandas.pydata.org/docs/reference/api/pandas.Series.quantile.html
print(med_inc.quantile(.25))
print(med_inc.quantile([0.25,0.75]))

# print(med_inc.size)

# get Dataframes to apply mean after
# pct_25 = med_inc.size * .25 # initial approach but found quantile as I tried to work it out
q1_25_float = med_inc.quantile(.25)
q3_75_float = med_inc.quantile(.75)

print('calc quartiles per quantile floats')
# df.head(pct_25)
# ref @ https://pandas.pydata.org/docs/user_guide/10min.html#boolean-indexing
# gets all rows where the column values of column MedInc is within 0 - 25% quartile
df_q1_bottom = df[med_inc <= q1_25_float] 
# gets all rows where the column values of column MedInc is within 75 - 100% quartile
df_q4_top = df[med_inc >= q3_75_float]

# apply mean to each quartile set
# ref @ https://pandas.pydata.org/docs/reference/api/pandas.Series.mean.html
# get Series/vector per column MedHouseVal and apply mean; repeat for each quartile
q1_bottom_mean = df_q1_bottom['MedHouseVal'].mean()
q4_top_mean = df_q4_top['MedHouseVal'].mean()

print(f'q1_bottom_mean: {q1_bottom_mean} | q4_top_mean: {q4_top_mean}')

# absolute delta between mean values where \top - bottom\
abs_mean_delta = abs(q4_top_mean - q1_bottom_mean)
print(f'abs_mean_delta: {abs_mean_delta}')

# round to nearest 3 decimals
q2_income_value_gap = round(abs_mean_delta, 3)

print(f'q2_income_value_gap: {q2_income_value_gap}')


# your code here
# raise NotImplementedError

             MedInc      HouseAge      AveRooms     AveBedrms    Population  \
count  20640.000000  20640.000000  20640.000000  20640.000000  20640.000000   
mean       3.870671     28.639486      5.429000      1.096675   1425.476744   
std        1.899822     12.585558      2.474173      0.473911   1132.462122   
min        0.499900      1.000000      0.846154      0.333333      3.000000   
25%        2.563400     18.000000      4.440716      1.006079    787.000000   
50%        3.534800     29.000000      5.229129      1.048780   1166.000000   
75%        4.743250     37.000000      6.052381      1.099526   1725.000000   
max       15.000100     52.000000    141.909091     34.066667  35682.000000   

           AveOccup      Latitude     Longitude   MedHouseVal  
count  20640.000000  20640.000000  20640.000000  20640.000000  
mean       3.070655     35.631861   -119.569704      2.068558  
std       10.386050      2.135952      2.003532      1.153956  
min        0.692308     32.54000

In [7]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Exercise 5.2
assert isinstance(q2_income_value_gap, float), "q2_income_value_gap must be a float"
assert q2_income_value_gap > 0, "Gap should be positive (top - bottom)"
assert q2_income_value_gap < 10.0, "Gap seems unreasonably large"
print(f"Income-Value gap: {q2_income_value_gap:.3f}")

Income-Value gap: 1.938


## 3. Train/test split *(10 points)*

Perform an 80 / 20 split of the predictors ``X`` (all columns except ``MedHouseVal``) and the
target ``y`` (only ``MedHouseVal``).  Use ``random_state=RANDOM_STATE``.  Store the **row counts**
of each split in a tuple **`q3_split_counts = (n_train, n_test)`**.

In [8]:

# apply 80/20 split on Dataframe
# ref @ https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.quantile.html
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html
# apply to all columns but MedHouseVal
# df_copy = df.copy()
# df_all_pred_columns = df_copy.drop()
# df_80_float = df.quantile(0.8, method="table", interpolation="nearest")

# use sklearn split...

# get pd.Series/Vector for predictors
# ref @ https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html
X = df.drop(columns=['MedHouseVal'])

# get pd.Series/Vector for target
y = df['MedHouseVal']

# split 80/20
# per https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

# row counts for X, y
# print(X_train[::1])
# print(X_train['MedInc'].size)
n_train = X_train['MedInc'].size
n_test = X_test['MedInc'].size

q3_split_counts = (n_train, n_test)
print(f'q3_split_counts: {q3_split_counts}')

# your code here
# raise NotImplementedError

q3_split_counts: (16512, 4128)


In [9]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Exercise 3
assert isinstance(q3_split_counts, tuple), "q3_split_counts must be a tuple"
assert len(q3_split_counts) == 2, "q3_split_counts should have 2 elements (train, test)"
assert all(
    isinstance(n, (int, np.integer)) for n in q3_split_counts
), "Both counts must be integers"
assert all(n > 0 for n in q3_split_counts), "Both counts must be positive"
print(f"Train/Test split: {q3_split_counts[0]} / {q3_split_counts[1]}")

Train/Test split: 16512 / 4128


## 4. k-NN with *k = 5* *(10 points)*

Train a ``KNeighborsRegressor`` with ``n_neighbors=5`` on the training data and compute the **test
RMSE**.  Store the scalar in **`q4_knn5_rmse`** rounded to **three** decimals.

In [10]:

# ref @ https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html#sklearn.neighbors.KNeighborsRegressor
neigh = KNeighborsRegressor(n_neighbors=5)
neigh.fit(X_train, y_train)
y_pred = neigh.predict(X_test)

# check MSE per imports not RMSE, Idk why but seems odd
# ref @ https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html
mse = mean_squared_error(y_test, y_pred)
print(f'mse: {mse}')

# per https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html
# but top cell said not to edit the imports so I have to assume i apply the square root manually
# rmse = root_mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print(f'rmse: {rmse}')

q4_knn5_rmse = round(rmse,3)
print(f'q4_knn5_rmse: {q4_knn5_rmse}')

# your code here
# raise NotImplementedError

mse: 1.1186823858768293
rmse: 1.0576778270706204
q4_knn5_rmse: 1.058


In [11]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Exercise 5.4
assert isinstance(q4_knn5_rmse, float), "RMSE must be a float"
assert q4_knn5_rmse >= 0, "RMSE must be non-negative"
assert q4_knn5_rmse < 10.0, "RMSE seems unreasonably large"
print(f"k-NN (k=5) Test RMSE: {q4_knn5_rmse:.3f}")

k-NN (k=5) Test RMSE: 1.058


## 5. Reusable k-NN evaluator *(15 points)*

Implement a function **`knn_rmse(k: int) -> float`** that
1. trains a ``KNeighborsRegressor`` with the given *k* on **`X_train`, `y_train`** (from Exercise 5.3),
2. returns the **test RMSE** rounded to three decimals.

In [12]:

def knn_rmse(k: int) -> float:
    neigh = KNeighborsRegressor(n_neighbors=k)
    neigh.fit(X_train, y_train)
    y_pred = neigh.predict(X_test)

    # check MSE per imports not RMSE, Idk why but seems like a typo??
    # ref @ https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html
    mse = mean_squared_error(y_test, y_pred)
    print(f'mse: {mse}')
    
    # per https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html
    # but top cell said not to edit the imports so I have to assume i apply the square root manually
    # rmse = root_mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    print(f'rmse: {rmse}')

    q4_knn5_rmse = round(rmse,3)
#     print(f'q4_knn5_rmse: {q4_knn5_rmse}')
    return q4_knn5_rmse

# your code here
# raise NotImplementedError

In [13]:
# If all tests pass (there might be hidden tests), you will earn 15 points
# Test Cell: Exercise 5.5
assert callable(knn_rmse), "knn_rmse should be a callable function"
# Test with k=5
test_rmse = knn_rmse(5)
assert isinstance(test_rmse, float), "knn_rmse should return a float"
assert test_rmse >= 0, "RMSE must be non-negative"
assert test_rmse < 10.0, "RMSE seems unreasonably large"
print(f"knn_rmse(5) = {test_rmse:.3f}")

mse: 1.1186823858768293
rmse: 1.0576778270706204
knn_rmse(5) = 1.058


## 6. Linear Regression baseline *(15 points)*

Fit an ordinary least-squares ``LinearRegression`` model (with default settings) on the training
data and compute its **test RMSE**.  Store the value in **`q6_linreg_rmse`** rounded to three
decimals.

In [14]:

# OLR not covered in module 1 reading or content but found docs for ref @ 
# https://scikit-learn.org/stable/auto_examples/linear_model/plot_ols_ridge.html
regressor = LinearRegression().fit(X_train, y_train)
y_pred = regressor.predict(X_test)

# check MSE per imports not RMSE, Idk why but seems like a typo??
# ref @ https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html
mse = mean_squared_error(y_test, y_pred)
print(f'mse: {mse}')
print(f"Mean squared error: {mse:.2f}")

# per https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html
# but top cell said not to edit the imports so I have to assume i apply the square root manually
# rmse = root_mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print(f'rmse: {rmse}')


q6_linreg_rmse = round(rmse,3)
print(f'q4_knn5_rmse: {q6_linreg_rmse}')

# your code here
# raise NotImplementedError

mse: 0.5558915986952443
Mean squared error: 0.56
rmse: 0.7455813830127763
q4_knn5_rmse: 0.746


In [15]:
# If all tests pass (there might be hidden tests), you will earn 15 points
# Test Cell: Exercise 6
assert isinstance(q6_linreg_rmse, float), "RMSE must be a float"
assert q6_linreg_rmse >= 0, "RMSE must be non-negative"
assert q6_linreg_rmse < 10.0, "RMSE seems unreasonably large"
print(f"Linear Regression Test RMSE: {q6_linreg_rmse:.3f}")

Linear Regression Test RMSE: 0.746


## 7. 5-fold CV for multiple *k* values *(15 points)*

Implement **`cross_val_knn(k_values: List[int]) -> Dict[int, float]`** that performs
5-fold cross-validation *only on the training split* for each *k* in `k_values` and returns a
dictionary mapping *k* → mean CV RMSE (rounded to three decimals).  Use
``KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)``.

In [16]:

# Debugging...
neigh = KNeighborsRegressor(n_neighbors=1)
kf = KFold(n_splits=5, random_state=RANDOM_STATE, shuffle=True)
print(f'kf: {kf}')
for i, (train_index, test_index) in enumerate(kf.split(X)):
    print(f"Fold {i}:")
    print(f"  Train: index={train_index}")
    print(f"  Test:  index={test_index}")
    
    # get just the first fold to get nD arrays for train and test
    if i == 0:
        # got error getting this. [] was not taking indices
#       X_train_kf, X_test_kf = X[train_index], X[test_index] 
#       works with iloc per https://pandas.pydata.org/docs/user_guide/indexing.html#different-choices-for-indexing
        X_train_kf, X_test_kf = X.iloc[train_index], X.iloc[test_index] 
        y_train_kf, y_test_kf = y[train_index], y[test_index]
        print(f'X_train_kf.size: {X_train_kf.size}')
        print(f'X_test_kf.size: {X_test_kf.size}')
        print(f'y_train_kf.size: {y_train_kf.size}')
        print(f'y_test_kf.size: {y_test_kf.size}')
        
# per https://scikit-learn.org/stable/modules/model_evaluation.html#string-name-scorers
# i experimented with setting the mse to the cv but not clear why since it thought it did. 
# not sure what its calculating then since default it None??
# the neg_ is odd but seems to be by design per doc "higher return values are better than lower return values."
# so i guess i interpret as [-mse] is better as it get closer to 0 so in latex 
# e.g. cv_scores_list_per_kf: [-1.54820515 -1.65067089 -1.50666375 -1.56568077 -1.6664944 ]
# display(Math(r'MAE_n \le 0 \text{ or any negative } MAE \in \mathbb{Z}^- \mid -MAE_0 > -MAE_1 \mid MAE_0 \gg MAE_1 \mid MAE_0 \text{ better than } MAE_1'))
cv_scores_list_per_kf_1 = cross_val_score(neigh, X, y, cv=kf)
print(f'cv_scores_list_per_kf_1: {cv_scores_list_per_kf_1}')
        
cv_scores_list_per_kf = cross_val_score(neigh, X, y, cv=kf, scoring='neg_mean_squared_error')
print(f'cv_scores_list_per_kf: {cv_scores_list_per_kf}')

cv_scores_list_per_kf_2 = cross_val_score(neigh, X, y, cv=kf, scoring='neg_root_mean_squared_error')
print(f'cv_scores_list_per_kf_2: {cv_scores_list_per_kf_2}')

print(f'cv_scores_list_per_kf type: {type(cv_scores_list_per_kf)}')

print(f'cv_scores_list_per_kf mean: {abs(cv_scores_list_per_kf.mean())}')

print(f'cv_scores_list_per_kf sqrt mean: {np.sqrt(abs(cv_scores_list_per_kf.mean()))}')

# Debugging...END


def cross_val_knn(k_values: List[int]) -> Dict[int, float]:
    mapping = {}
    for k in k_values:
        neigh = KNeighborsRegressor(n_neighbors=k)
        kf = KFold(n_splits=5, random_state=RANDOM_STATE, shuffle=True)
#         cv_scores_list_per_kf = cross_val_score(neigh, X, y, cv=kf, scoring='neg_mean_squared_error')
        cv_scores_list_per_kf = cross_val_score(neigh, X, y, cv=kf, scoring='neg_root_mean_squared_error')
#         mse = mean_squared_error(y_test, y_pred)
# get mean of cv scores and its absolute so it can be interpreted as a positive correctly
# per https://numpy.org/doc/stable/reference/generated/numpy.ndarray.mean.html#numpy.ndarray.mean
        mean_rmse = abs(cv_scores_list_per_kf.mean())
#     get square root
#     per https://numpy.org/doc/stable/reference/generated/numpy.sqrt.html
#         realized the mean and square root felt forced so found neg_root_mean_squared_error 
#         rmean_mse = np.sqrt(mse)
        mean_rmse_rd = round(mean_rmse, 3)
        print(f'rmse: {mean_rmse}')
        print(f'rmse_rd: {mean_rmse_rd}')
#         cv_mse = 0

#         q4_knn5_rmse = round(mse,3)
        mapping[k] = mean_rmse_rd
#         realized the mean and square root felt forced so found neg_root_mean_squared_error 
    return mapping 

# check
test_vals = cross_val_knn([3, 5])
test_vals
# check end

# your code here
# raise NotImplementedError

kf: KFold(n_splits=5, random_state=42, shuffle=True)
Fold 0:
  Train: index=[    1     2     4 ... 20636 20637 20639]
  Test:  index=[    0     3     6 ... 20633 20635 20638]
X_train_kf.size: 132096
X_test_kf.size: 33024
y_train_kf.size: 16512
y_test_kf.size: 4128
Fold 1:
  Train: index=[    0     1     2 ... 20637 20638 20639]
  Test:  index=[    5     8    14 ... 20628 20630 20636]
Fold 2:
  Train: index=[    0     1     3 ... 20637 20638 20639]
  Test:  index=[    2     7    10 ... 20613 20615 20620]
Fold 3:
  Train: index=[    0     2     3 ... 20637 20638 20639]
  Test:  index=[    1     4    12 ... 20629 20631 20634]
Fold 4:
  Train: index=[    0     1     2 ... 20635 20636 20638]
  Test:  index=[    9    11    13 ... 20632 20637 20639]
cv_scores_list_per_kf_1: [-0.18146714 -0.20831424 -0.15803505 -0.17456132 -0.2381204 ]
cv_scores_list_per_kf: [-1.54820515 -1.65067089 -1.50666375 -1.56568077 -1.6664944 ]
cv_scores_list_per_kf_2: [-1.24426892 -1.28478438 -1.22746232 -1.25127166 -

{3: 1.085, 5: 1.061}

In [17]:
# If all tests pass (there might be hidden tests), you will earn 15 points
# Test Cell: Exercise 7
test_vals = cross_val_knn([3, 5])
assert callable(cross_val_knn), "cross_val_knn should be a callable function"
assert isinstance(test_vals, dict), "cross_val_knn should return a dictionary"
assert len(test_vals) == 2, "Dictionary should have one entry per k value"
assert all(isinstance(k, int) for k in test_vals.keys()), "Keys should be integers"
assert all(isinstance(v, float) for v in test_vals.values()), "Values should be floats"
assert all(v >= 0 for v in test_vals.values()), "RMSE values must be non-negative"
print(f"Sample CV scores: {test_vals}")

rmse: 1.0847750110777026
rmse_rd: 1.085
rmse: 1.0605853615955088
rmse_rd: 1.061
Sample CV scores: {3: 1.085, 5: 1.061}


## 8. Choose best *k* and evaluate on test set *(15 points)*

1. Use your `cross_val_knn` from Exercise 5.7 with the list `[1, 3, 5, 7, 9, 15, 25]`.
2. Identify the *k* that achieves the **lowest** cross-validated RMSE → **`q8_best_k`
3. Finally, use your `knn_rmse` function from Exercise 5.5 to compute the **test set RMSE** for this best *k* → **`q8_test_rmse`**

In [18]:

test_vals = cross_val_knn([1, 3, 5, 7, 9, 15, 25])
keys_list = list(test_vals.keys())
values_list = list(test_vals.values())
# get index of lowest and select k from it
# no sorting to maintain the indices
# per https://docs.python.org/3/library/functions.html#min
# picks the first from dups by default
min_val = min(test_vals.values()) 
lowest_value_index = values_list.index(min_val)
q8_best_k = keys_list[lowest_value_index]

print(values_list)
print(f'min_val: {min_val}')

# np attempt
# print(np.min(test_vals.values(),axis=0))

# loop attempt
# for i, (k, mse) in enumerate(test_vals):

# get index of values
print(f'lowest_value_index: {lowest_value_index}')

# get key from index
print(keys_list)
print(f'q8_best_k: {q8_best_k}')
    

# run knn on k
q8_test_rmse = knn_rmse(q8_best_k)
print(f'q8_test_rmse: {q8_test_rmse}')


# your code here
# raise NotImplementedError

rmse: 1.2597430021261835
rmse_rd: 1.26
rmse: 1.0847750110777026
rmse_rd: 1.085
rmse: 1.0605853615955088
rmse_rd: 1.061
rmse: 1.05527818760799
rmse_rd: 1.055
rmse: 1.0547841723265292
rmse_rd: 1.055
rmse: 1.0639121535538103
rmse_rd: 1.064
rmse: 1.0827937988252565
rmse_rd: 1.083
[1.26, 1.085, 1.061, 1.055, 1.055, 1.064, 1.083]
min_val: 1.055
lowest_value_index: 3
[1, 3, 5, 7, 9, 15, 25]
q8_best_k: 7
mse: 1.1045381245194705
rmse: 1.0509700873571381
q8_test_rmse: 1.051


In [19]:
# If all tests pass (there might be hidden tests), you will earn 15 points
# Test Cell: Exercise 8
k_candidates = [1, 3, 5, 7, 9, 15, 25]
assert isinstance(q8_best_k, int), "q8_best_k must be an integer"
assert q8_best_k > 0, "k must be positive"
assert isinstance(q8_test_rmse, float), "q8_test_rmse must be a float"
assert q8_test_rmse >= 0, "RMSE must be non-negative"
assert q8_test_rmse < 10.0, "RMSE seems unreasonably large"
print(f"Best k: {q8_best_k}, Test RMSE: {q8_test_rmse:.3f}")

Best k: 7, Test RMSE: 1.051


## Next Steps

Congratulations on completing the assignment! Before submitting:

1. Make sure all your cells run without errors.
2. Ensure you've answered all parts of each question.
3. If any autograder tests fail, revisit your answers.
